In [1]:
# !pip install umap-learn
# !pip install tensorflow
# !pip install mne
# !pip install torch
# !pip install pandas
# !pip install scipy
# !pip install tensorflow-gpu==2.10.0

In [1]:
import os
import mne

In [2]:
import tensorflow as tf
import numpy as np
import random
import torch
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import disable_eager_execution


from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
# import seaborn as sns
# import plotly.express as px
import umap.umap_ as umap
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer

from keras.utils import to_categorical
import warnings
from keras.layers import Input, Dense, Lambda, Reshape, Flatten, BatchNormalization, LeakyReLU, Dropout
from keras.layers import concatenate as concat
from keras.models import Model
import tensorflow.keras.backend as K
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

# disable_eager_execution()
# print(tf.executing_eagerly())

random_seed=3
random.seed(random_seed)

c:\Users\carlo\miniconda3\envs\tensorflow\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# print(tf.config.list_physical_devices('GPU'))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

# print("CUDA disponível: ", torch.cuda.is_available())
# print("Nome da GPU: ", torch.cuda.get_device_name(0))

Num GPUs Available:  1


In [4]:
classes = ['abnormal', 'normal']

# define as classes (normal ou abnormal)
y = []

# lista contendo os exames
raws = []

for classe in classes:
  folder_path = "Dataset/tuh_eeg_abnormal/edf/train/" + classe
  files = os.listdir(folder_path)

  # folder_path = "/content/drive/MyDrive/IC/AVC/Dataset/tuh_eeg_abnormal/edf/train/abnormal/01_tcp_ar"
  # files = os.listdir(folder_path)

  for file in files[0:1]:
      path = folder_path + '/' + file
      # classe = folder_path.split('/')[-2]
      if(classe == 'abnormal'):
          y.append(0)
      elif(classe == 'normal'):
          y.append(1)

      raws.append(mne.io.read_raw_edf(path, preload=True))
  
  print(len(y))
#   print(y)

Extracting EDF parameters from c:\Users\carlo\Desktop\EEG_Analysis\EEG_Analysis-for-CVA\Dataset\tuh_eeg_abnormal\edf\train\abnormal\aaaaaaaq_s004_t000.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 338999  =      0.000 ...  1355.996 secs...
1
Extracting EDF parameters from c:\Users\carlo\Desktop\EEG_Analysis\EEG_Analysis-for-CVA\Dataset\tuh_eeg_abnormal\edf\train\normal\aaaaaaav_s004_t000.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 316999  =      0.000 ...  1267.996 secs...
2


In [6]:
# # informações sobre as amostras
# raw = raws[9]
# raw.info

In [7]:
# # visualização dos nomes dos canais
# raw.ch_names

In [5]:
tempos_amostragem = []
for raw in raws:
    tempo_final = raw.times[-1]
    tempos_amostragem.append(tempo_final)

tempo_minimo = min(tempos_amostragem)

# print(tempos_amostragem)
print(tempo_minimo)

1267.996


In [6]:
# para selecionar somente alguns canais específicos
# raw.pick(['EEG F3-REF', 'EEG T1-REF'])

# eeg_and_eog = raw.copy().drop_channels(["EMG-REF", "PHOTIC-REF", "IBI", "BURSTS", "SUPPR"])
# print(len(raw.ch_names), "→", len(eeg_and_eog.ch_names))

# raw.pick(['EEG F3-REF', 'EEG T1-REF'])

eeg_set = []
for raw in raws:
    try:
        # eeg = raw.copy().drop_channels(["EMG-REF", "PHOTIC-REF", "IBI", "BURSTS", "SUPPR"])
        eeg = raw.copy().crop(0,tempo_minimo).pick(['EEG FP1-REF','EEG FP2-REF','EEG F3-REF','EEG F4-REF','EEG C3-REF','EEG C4-REF','EEG P3-REF','EEG P4-REF','EEG O1-REF','EEG O2-REF','EEG F7-REF','EEG F8-REF','EEG T3-REF','EEG T4-REF','EEG T5-REF','EEG T6-REF','EEG A1-REF','EEG A2-REF','EEG FZ-REF','EEG CZ-REF','EEG PZ-REF'])
        eeg_set.append(eeg)
        print(len(raw.ch_names), "→", len(eeg.ch_names))

    except ValueError:
        pass

30 → 21
30 → 21


In [13]:
eeg = eeg_set[0]
eeg.info
eeg.times


array([0.000000e+00, 4.000000e-03, 8.000000e-03, ..., 1.267988e+03,
       1.267992e+03, 1.267996e+03])

In [10]:
# # quantidade de amostras e tempo de amostragem
print('Nº amostras do EEG1 :', len(eeg_set[0].times),'; Tempo de amostragem: ',eeg_set[0].times[-1],'s' )
# print('Nº amostras do EEG2 :', len(eeg_set[1].times),'; Tempo de amostragem: ',eeg_set[1].times[-1],'s' )
# print('Nº amostras do EEG3 :', len(eeg_set[2].times),'; Tempo de amostragem: ',eeg_set[2].times[-1],'s' )
# print('Nº amostras do EEG4 :', len(eeg_set[3].times),'; Tempo de amostragem: ',eeg_set[3].times[-1],'s' )
# print('Nº amostras do EEG5 :', len(eeg_set[4].times),'; Tempo de amostragem: ',eeg_set[4].times[-1],'s' )
# print('Nº amostras do EEG6 :', len(eeg_set[5].times),'; Tempo de amostragem: ',eeg_set[5].times[-1],'s' )
# print('Nº amostras do EEG7 :', len(eeg_set[6].times),'; Tempo de amostragem: ',eeg_set[6].times[-1],'s' )


Nº amostras do EEG1 : 317000 ; Tempo de amostragem:  1267.996 s


In [7]:
# define o número de amostras de cada canal do EEG a ser considerado
n_amostras = int(tempo_minimo/0.004)
print(n_amostras)
if(tempo_minimo % 0.004) > 0:
  n_amostras += 1
print(n_amostras)

316999
317000


In [8]:
import matplotlib.pyplot as plt
import numpy as np

###################################################################
## código de teste

# samples = np.array(raw[0][0])

# for i in range(1, 36):
#     # print(raw[i][0])
#     samples = np.vstack([samples, raw[i][0]])

# # print(samples.shape)

# plt.plot(samples[0])
# plt.plot(samples[2])

###################################################################


X = []
# j=0
for eeg in eeg_set:

    x = np.array(eeg[0][0][0][:n_amostras])

    for i in range(1, 21):
        # x = np.vstack([x, eeg[i][0]])
        x = np.append(x, eeg[i][0][0][:n_amostras])
        # print('n de amostras:',i,(eeg[i]))

    # plt.plot(x[0])
    # plt.plot(x[2])
    # plt.plot(x)
    # print(j, ':', len(x))
    # j+=1

    X.append(x)

X = np.array(X)

In [9]:
X.shape

(2, 6657000)

In [13]:
# print(len(X[0]))
# print(len(X[2]))
# print(len(X[7]))
# print(len(X[12]))

# # print(len(eeg_set[3].times))
# plt.plot(X[0,:])
# plt.plot(X[2,:])
# plt.plot(X[7,:])
# plt.plot(X[12,:])

In [14]:
###################################################################
# scale data
# scaler = StandardScaler().fit(samples.T)
# #scaler = MinMaxScaler().fit(X.T)
# samplesNorm = scaler.transform(samples.T).T

###################################################################
# samplesNorm = []
# # scale data
# for X in samples:
#     # print(X)
#     X_reshape = X.reshape(-1,1)
#     # print(X_reshape)
#     # scaler = StandardScaler().fit(X_reshape)
#     scaler = MinMaxScaler().fit(X_reshape)
#     Xnorm_reshape = scaler.transform(X_reshape)
#     # print(Xnorm_reshape)
#     Xnorm = np.squeeze(Xnorm_reshape)
#     # print(Xnorm)

#     samplesNorm.append(Xnorm)
###################################################################


# scale data
#scaler = StandardScaler().fit(X.T)
scaler = MinMaxScaler().fit(X.T)
Xnorm = scaler.transform(X.T).T

In [15]:
# # for Xnorm in samplesNorm:
#     # plt.plot(Xnorm[0])
#     # plt.plot(Xnorm[2])
#     # plt.plot(Xnorm)
# plt.plot(Xnorm[0,:])
# plt.plot(Xnorm[2,:])
# plt.plot(Xnorm[7,:])
# plt.plot(Xnorm[12,:])

In [16]:
#classes
y = np.array(y)

## Conditional VAE - Keras

In [17]:
# X_train, X_val, y_train, y_val = train_test_split(Xnew, y, test_size=0.1, random_state=42)

In [18]:
X_train = X
y_train = y

print(X_train.shape)
print(y_train.shape)

# type(y_train)

(2, 6657000)
(2,)


In [19]:
warnings.filterwarnings('ignore')
%pylab inline

hidden_dim = 2 # latent space size
batch_size = 1 # batch size
optim = Adam(learning_rate=0.0001)


n_x = X_train.shape[1]
n_y = y_train.shape[0]


n_epoch = 50

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [20]:
# def sample_z(args):
#     mu, l_sigma = args
#     eps = K.random_normal(shape=(m, n_z), mean=0., stddev=1.)
#     return mu + K.exp(l_sigma / 2) * eps

In [21]:
def noiser(args):
    global mean, log_var
    mean, log_var = args
    batch = tf.shape(mean)[0]
    dim = tf.shape(mean)[1]
    N = K.random_normal(shape=(batch, dim), mean=0., stddev=1.0)
    return mean + K.exp(0.5*log_var) * N

def vae_loss(input, output):
    # compute the average MSE error, then scale it up i.e. simply sum on all axes
    reconstruction_loss = K.sum(K.square(output-input))
    #reconstruction_loss = K.sum(K.binary_crossentropy(y, x), axis=-1)
    # compute the KL loss
    kl_loss = -0.5 * K.sum(1 + log_var - K.square(mean) - K.square(K.exp(log_var)), axis=-1)
    # return the average loss over all images in batch
    total_loss = reconstruction_loss + kl_loss
    return total_loss

def dropout_and_batchnorm(x):
    return Dropout(0.3)(BatchNormalization()(x))

In [22]:
# Encoder
#X = Input(shape=(n_x,))
#label = Input(shape=(n_y,))

inp = Input(batch_shape=(batch_size, n_x))
#fl = Flatten()(inp)
lb = Input(shape=(1,))
x = concat([inp, lb])
x = Dense(64, activation="relu")(x)
x = dropout_and_batchnorm(x)
x = Dense(32, activation="relu")(x)
x = dropout_and_batchnorm(x)
x = Dense(32, activation="relu")(x)
shape_before_flattening = K.int_shape(x)[1:]  # the decoder will need this!

In [23]:
# Latent Space
x = Flatten()(x)
mean = Dense(hidden_dim)(x)
log_var = Dense(hidden_dim)(x)
h = Lambda(noiser, output_shape=(hidden_dim,), name="latent_space")([mean, log_var])

In [24]:
# Decoder
input_dec = Input(shape=(hidden_dim,), name="decoder_input")
lb_dec = Input(shape=(1,)) #1 = size of y (regression)
d = concat([input_dec, lb_dec])
d = Dense(np.prod(shape_before_flattening))(d)
d = Reshape(shape_before_flattening)(d)
d = Dense(32, activation="relu")(d)
d = dropout_and_batchnorm(d)
d = Dense(32, activation="relu")(d)
d = dropout_and_batchnorm(d)
d = Dense(64, activation="relu")(d)
#decoded = Dense(n_x, activation=LeakyReLU())(d) #n_x = size output transform
decoded = Dense(n_x, activation="sigmoid")(d) #n_x = size output transform

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2]

In [ ]:
# def vae_loss(input, output):
#     # compute the average MSE error, then scale it up i.e. simply sum on all axes
#     reconstruction_loss = K.sum(K.square(output-input))
#     #reconstruction_loss = K.sum(K.binary_crossentropy(y, x), axis=-1)
#     # compute the KL loss
#     kl_loss = -0.5 * K.sum(1 + log_var - K.square(mean) - K.square(K.exp(log_var)), axis=-1)
#     # return the average loss over all images in batch
#     total_loss = K.mean(reconstruction_loss + kl_loss)
#     return total_loss

In [ ]:
# CVAE
encoder = Model([inp, lb], h, name="encoder")
decoder = Model([input_dec, lb_dec], decoded, name="decoder")
cvae = Model(
    inputs=[inp, lb, lb_dec],
    outputs=decoder([encoder([inp, lb]), lb_dec]),
    name="cvae"
)

cvae.compile(optimizer=optim, loss=vae_loss)
cvae.summary()

In [ ]:
# tf.config.list_physical_devices('GPU')

In [ ]:
cvae.fit(
    [X_train, y_train, y_train], X_train,
    epochs=n_epoch,
    batch_size=batch_size,
    shuffle=True,
    validation_split=0.1,
    callbacks = [EarlyStopping(patience = 5)]
)

## Visualize Latent Space

In [ ]:
z_train = encoder.predict([X_train, y_train])
encodings= np.asarray(z_train)

plt.hist(z_train[2,:])
plt.xlabel('Espaço Latente')
plt.ylabel('Frequencia')
plt.show()

In [ ]:
# UMAP
x_reducer = umap.UMAP(n_components=2, metric='braycurtis',
                    min_dist=0.8, random_state=42)
X_emb = x_reducer.fit_transform(X_train)

plt.scatter(X_emb[:, 0], X_emb[:, 1], c=y_train)

In [ ]:
# Espaço latente reduzido com UMAP

z_reducer = umap.UMAP(n_components=2, metric='braycurtis',
                    min_dist=0.8, random_state=42)
z_emb = z_reducer.fit_transform(z_train)

plt.scatter(z_emb[:, 0], z_emb[:, 1], c=y_train)
plt.xlabel('UMAP-2')
plt.ylabel('UMAP-1')

cbar = colorbar()
cbar.solids.set_edgecolor("face")
draw()

In [ ]:
# Espaço latente reduzido

plt.scatter(z_train[:, 0], z_train[:, 1], c=y_train)
plt.xlabel('Latent-1')
plt.ylabel('Latent-2')

cbar = colorbar()
cbar.solids.set_edgecolor("face")
draw()

In [ ]:
x_reducer_3d = umap.UMAP(n_components=3, metric='braycurtis', min_dist=0.8, random_state=42)
embedding = x_reducer_3d.fit_transform(X_train)
df = pd.DataFrame(embedding, columns=['UMAP-1','UMAP-2','UMAP-3'])

fig = px.scatter_3d(
    df, x="UMAP-1", y="UMAP-2", z="UMAP-3",
    color=y_train
)
fig.update_traces(marker_size=8)
fig.show()


## Generate New Data

In [ ]:
max_y = max(y)
min_y = min(y)

In [ ]:
max_y

In [ ]:
min_y

In [ ]:
mu, sigma = 0, 1.0 # mean and standard deviation
z_new_sample = np.random.normal(mu, sigma, (1,100))
y_new_sample = np.random.uniform(min_y, max_y, (1,))
print(y_new_sample)

x_decoded = decoder.predict([z_new_sample, y_new_sample])

In [ ]:
fig = plt.figure(figsize = (10, 7))

plt.plot(x_decoded[0,:])
plt.plot(Xnew[1,:])

In [ ]:
random.seed(3)

tam_new_samples = 100
mu, sigma = 0, 1.0 # mean and standard deviation
z_new_sample = np.random.normal(mu, sigma, (tam_new_samples, hidden_dim))
y_new_sample = np.random.uniform(min_y, max_y, (tam_new_samples,))

X_dec = np.empty(shape = (tam_new_samples,1901))
y_dec = np.empty(shape = (tam_new_samples,1))

for i in range(tam_new_samples):
  y_dec[i] = y_new_sample[i,]
  #print(y_new_sample)

  x_decoded = decoder.predict([z_new_sample[i,:].reshape(1, -1), y_new_sample[i,].reshape(1,)])
  X_dec[i,: ] =  x_decoded


In [ ]:
y_new_sample[1]

In [ ]:
X_aug = np.append(X_train, X_dec, axis=0)
y_aug = np.append(y_train, y_dec)

In [ ]:
fig = plt.figure(figsize = (10, 7))

plt.plot(X_dec[1,: ])
plt.plot(X_dec[33,: ])
plt.plot(X_dec[5,: ])
plt.plot(X_dec[23,: ])
plt.plot(Xnew[1,:])

plt.legend(['Amostra gerada: B2=1,824', 'Amostra gerada: B2=1,777', 'Amostra gerada: B2=1,823', 'Amostra gerada: B2=1,754', 'Amostra original: B2=1,602'])
plt.xlabel('#Amostras')
plt.ylabel('Intensidade')

In [ ]:
# UMAP

X_emb_aug = x_reducer.transform(X_aug)
X_emb_dec = x_reducer.transform(X_dec)

#plt.scatter(X_emb[:, 0], X_emb[:, 1], c=y_train)
plt.scatter(X_emb[:, 0], X_emb[:, 1], marker='o',s=50)
plt.scatter(X_emb_dec[:, 0], X_emb_dec[:, 1],  marker='x',s=50)
#plt.scatter(X_emb_dec[:, 0], X_emb_dec[:, 1],  c=y_dec)

plt.legend(['Original','Gerado'])
plt.xlabel('UMAP-2')
plt.ylabel('UMAP-1')


### Algoritmos (Sem aumento de dados)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Define the parameters and their range
parameters = {'n_neighbors':np.arange(2, 30, 1)}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_knn= RandomizedSearchCV(KNeighborsRegressor(), parameters, scoring='r2', cv=10, random_state=23)
# Fit to the data
model_knn.fit(X_train, y_train)
#Get the optimised value of alpha
print('Best parameter n_neighbors = ', model_knn.best_params_['n_neighbors'])
print('R2 calibration: %5.3f'  % model_knn.score(X_train,y_train))
# Run a knn regression with the optimised value
model_knn1 = KNeighborsRegressor(n_neighbors=model_knn.best_params_['n_neighbors'])
y_cv = cross_val_predict(model_knn1, X_train, y_train, cv=10)
# y_cv=predicted
score_cv = r2_score(y_train, y_cv)
mse_cv = mean_squared_error(y_train, y_cv, squared=False)
print('R2 CV: %5.3f'  % score_cv)
print('RMSE CV: %5.3f' % mse_cv)

In [ ]:
# Teste
model_knn1.fit(X_train, y_train)
yhat_knn = model_knn1.predict(X_val)

print("RMSE:",mean_squared_error(yhat_knn, y_val, squared=False))
print("R2:",r2_score(yhat_knn, y_val))

In [ ]:
from sklearn.cross_decomposition import PLSRegression

# Define the parameters and their range
parameters = {'n_components':np.arange(2, 30, 1)}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_pls= RandomizedSearchCV(PLSRegression(), parameters, scoring='r2', cv=10, random_state=23)
# Fit to the data
model_pls.fit(X_train, y_train)
#Get the optimised value of alpha
print('Best parameter n_components = ', model_pls.best_params_['n_components'])
print('R2 calibration: %5.3f'  % model_pls.score(X_train,y_train))
# Run a ridge regression with the optimised value
model_pls1 = PLSRegression(n_components=model_pls.best_params_['n_components'])
y_cv = cross_val_predict(model_pls1, X_train, y_train, cv=10)
# y_cv=predicted
score_cv = r2_score(y_train, y_cv)
mse_cv = mean_squared_error(y_train, y_cv, squared=False)
print('R2 CV: %5.3f'  % score_cv)
print('RMSE CV: %5.3f' % mse_cv)

In [ ]:
# Teste
model_pls1.fit(X_train, y_train)
yhat_pls = model_pls1.predict(X_val)
print("RMSE:",mean_squared_error(yhat_pls,y_val,squared=False))

In [ ]:
from sklearn.svm import SVR
from scipy import stats

# Define the parameters and their range
parameters = {"C": stats.uniform(0.1,100), "epsilon": stats.expon(scale=.1),
              "kernel": ['rbf', 'linear', 'sigmoid']}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_svr = RandomizedSearchCV(SVR(gamma='auto'), parameters, scoring='r2', cv=10, random_state=42)
# Fit to the data
model_svr.fit(X_train, y_train)
#Get the optimised value of alpha
print('Best parameter epsilon = ', model_svr.best_params_['epsilon'])
print('Best parameter C = ', model_svr.best_params_['C'])
print('Best parameter kernel = ', model_svr.best_params_['kernel'])
#print('Best parameter gamma = ', model_svr.best_params_['gamma'])
print('R2 calibration: %5.3f'  % model_svr.score(X_train,y_train))
# Run a ridge regression with the optimised value
model_svr1 = SVR(C=model_svr.best_params_['C'], kernel=model_svr.best_params_['kernel'],
                 epsilon=model_svr.best_params_['epsilon'])
y_cv = cross_val_predict(model_svr1, X_train, y_train, cv=10)
# y_cv=predicted
score_cv = r2_score(y_train, y_cv)
mse_cv = mean_squared_error(y_train, y_cv, squared=False)
print('R2 CV (SVR): %5.3f'  % score_cv)
print('RMSE CV (SVR): %5.3f' % mse_cv)

In [ ]:
# Teste
model_svr1.fit(X_train, y_train)
yhat_svr = model_svr1.predict(X_val)
print("RMSE:",mean_squared_error(yhat_svr,y_val,squared=False))

### Algoritmos (Com aumento de dados)

In [ ]:
# Define the parameters and their range
parameters = {'n_neighbors':np.arange(2, 30, 1)}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_knn= RandomizedSearchCV(KNeighborsRegressor(), parameters, scoring='r2', cv=10, random_state=23)
# Fit to the data
model_knn.fit(X_aug, y_aug)
#Get the optimised value of alpha
print('Best parameter n_neighbors = ', model_knn.best_params_['n_neighbors'])
print('R2 calibration: %5.3f'  % model_knn.score(X_aug, y_aug))
# Run a knn regression with the optimised value
model_knn1 = KNeighborsRegressor(n_neighbors=model_knn.best_params_['n_neighbors'])
y_cv = cross_val_predict(model_knn1, X_aug, y_aug, cv=10)
# y_cv=predicted
score_cv = r2_score(y_aug, y_cv)
mse_cv = mean_squared_error(y_aug, y_cv, squared=False)
print('R2 CV: %5.3f'  % score_cv)
print('RMSE CV: %5.3f' % mse_cv)

In [ ]:
# Teste
model_knn1.fit(X_aug, y_aug)
yhat_knn = model_knn1.predict(X_val)

print("RMSE:",mean_squared_error(yhat_knn, y_val, squared=False))
print("R2:",r2_score(yhat_knn, y_val))

In [ ]:
# Define the parameters and their range
parameters = {'n_components':np.arange(2, 30, 1)}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_pls= RandomizedSearchCV(PLSRegression(), parameters, scoring='r2', cv=10, random_state=23)
# Fit to the data
model_pls.fit(X_aug, y_aug)
#Get the optimised value of alpha
print('Best parameter n_components = ', model_pls.best_params_['n_components'])
print('R2 calibration: %5.3f'  % model_pls.score(X_aug,y_aug))
# Run a PLS regression with the optimised value
model_pls1 = PLSRegression(n_components=model_pls.best_params_['n_components'])
y_cv = cross_val_predict(model_pls1, X_aug, y_aug, cv=10)
# y_cv=predicted
score_cv = r2_score(y_aug, y_cv)
mse_cv = mean_squared_error(y_aug, y_cv, squared=False)
print('R2 CV: %5.3f'  % score_cv)
print('RMSE CV: %5.3f' % mse_cv)

In [ ]:
# Teste
model_pls1.fit(X_aug, y_aug)
yhat_pls = model_pls1.predict(X_val)
print("RMSE:",mean_squared_error(yhat_pls,y_val,squared=False))

In [ ]:
# Define the parameters and their range
parameters = {"C": stats.uniform(0.1,100), "epsilon": stats.expon(scale=.1),
              "kernel": ['rbf']}
#Run a Grid search, using R^2 as the metric to optimise alpha
model_svr = RandomizedSearchCV(SVR(gamma='auto'), parameters, scoring='r2', cv=10, random_state=42)
# Fit to the data
model_svr.fit(X_aug, y_aug)
#Get the optimised value of alpha
print('Best parameter epsilon = ', model_svr.best_params_['epsilon'])
print('Best parameter C = ', model_svr.best_params_['C'])
print('Best parameter kernel = ', model_svr.best_params_['kernel'])
#print('Best parameter gamma = ', model_svr.best_params_['gamma'])
print('R2 calibration: %5.3f'  % model_svr.score(X_aug,y_aug))
# Run a ridge regression with the optimised value
model_svr1 = SVR(C=model_svr.best_params_['C'], kernel=model_svr.best_params_['kernel'],
                 epsilon=model_svr.best_params_['epsilon'])
y_cv = cross_val_predict(model_svr1, X_aug, y_aug, cv=10)
# y_cv=predicted
score_cv = r2_score(y_aug, y_cv)
mse_cv = mean_squared_error(y_aug, y_cv, squared=False)
print('R2 CV (SVR): %5.3f'  % score_cv)
print('RMSE CV (SVR): %5.3f' % mse_cv)

In [ ]:
# Teste
model_svr1.fit(X_aug, y_aug)
yhat_svr = model_svr1.predict(X_val)
print("RMSE:",mean_squared_error(yhat_svr,y_val,squared=False))